Get the CA Station Ids

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import requests

url = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json"
data = requests.get(url, params={"type": "met"}, timeout=60).json()

# 1) All CA met station objects (full JSON per station)
met_ca = [s for s in data.get("stations", []) if s.get("state") == "CA"]

# 2) Ensure 'id' exists and keep full JSON (already full objects)
#    Also extract/normalize 'shefcode'
ca_stations_full = []
for s in met_ca:
    station_id = s.get("id")
    shefcode = s.get("shefcode")
    if station_id:  # keep only stations with an id
        s = dict(s)  # copy so we can safely add/modify fields
        s["id"] = station_id
        s["shefcode"] = shefcode  # keep as-is (may be None)
        ca_stations_full.append(s)

# 3) Build lists / maps for retrieval using SHEF codes
#    - shefcodes list (only non-empty)
shefcodes = [s["shefcode"] for s in ca_stations_full if s.get("shefcode")]

#    - map shefcode -> full station json
by_shefcode = {s["shefcode"]: s for s in ca_stations_full if s.get("shefcode")}
by_id = {s["id"]: s for s in ca_stations_full if s.get("id")}

#    - map shefcode -> station id (handy if another endpoint needs id)
shefcode_to_id = {s["shefcode"]: s["id"] for s in ca_stations_full if s.get("shefcode")}

print("CA met station count (all):", len(ca_stations_full))
print("CA met station count (with shefcode):", len(shefcodes))
print("SHEF codes:", shefcodes)

# Example retrieval by SHEF code:
example_code = shefcodes[0] if shefcodes else None
if example_code:
    station = by_shefcode[example_code]
    print("\nExample lookup by SHEF:", example_code)
    print("Station id:", station["id"])
    print("Station name:", station.get("name"))

CA met station count (all): 35
CA met station count (with shefcode): 35
SHEF codes: ['SDBC1', 'LJAC1', 'AGXC1', 'OHBC1', 'PRJC1', 'PFDC1', 'PFXC1', 'PXAC1', 'BAXC1', 'PSXC1', 'ICAC1', 'NTBC1', 'PSLC1', 'MEYC1', 'FTPC1', 'PXSC1', 'PXOC1', 'RTYC1', 'AAMC1', 'LNDC1', 'OMHC1', 'OKXC1', 'OBXC1', 'PPXC1', 'RCMC1', 'PRYC1', 'MZXC1', 'PSBC1', 'UPBC1', 'DPXC1', 'PCOC1', 'ANVC1', 'HBYC1', 'NJLC1', 'CECC1']

Example lookup by SHEF: SDBC1
Station id: 9410170
Station name: San Diego


In [13]:
import json

output_file_path = "/content/drive/Shareddrives/CMPE_255/data/ca_stations_full.json"

with open(output_file_path, "w") as f:
    json.dump(ca_stations_full, f, indent=4)

print(f"'ca_stations_full' saved to {output_file_path}")

'ca_stations_full' saved to /content/drive/Shareddrives/CMPE_255/data/ca_stations_full.json


Get the winds information from the nbdc dataset

In [8]:
import os
import gzip
import requests

YEARS = range(2021, 2026)
BASE = "https://www.ndbc.noaa.gov/data/historical/stdmet"
OUT_DIR = "/content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025"
TIMEOUT = 60

os.makedirs(OUT_DIR, exist_ok=True)

def try_download(url: str) -> bytes | None:
    """Return raw bytes if download succeeds, else None."""
    try:
        r = requests.get(url, timeout=TIMEOUT, stream=True)
        if r.status_code != 200:
            return None
        return r.content
    except requests.RequestException:
        return None

def download_year_text(station: str, year: int) -> str | None:
    """
    Download gz file and return decompressed text (str),
    or None if not found / failed.
    """
    # NDBC filenames are commonly lowercase station ids; we try both.
    candidates = [
        f"{BASE}/{station.lower()}h{year}.txt.gz",
        f"{BASE}/{station.upper()}h{year}.txt.gz",
    ]
    for url in candidates:
        raw = try_download(url)
        if raw is None:
            continue
        try:
            return gzip.decompress(raw).decode("utf-8", errors="replace")
        except OSError:
            # Not a valid gzip or corrupted download
            return None
    return None

def combine_station(station: str, years=YEARS) -> tuple[str, int]:
    """
    Create a combined file for a station, return (output_path, years_found).
    Keeps the first header block (#...) only once.
    """
    combined_lines = []
    header_added = False
    years_found = 0

    for year in years:
        text = download_year_text(station, year)
        if not text:
            continue

        years_found += 1
        lines = text.splitlines(True)  # keep line endings

        # NDBC files usually have one or more header/comment lines starting with '#'
        # followed by data lines. Keep headers only for the first appended year.
        if not header_added:
            combined_lines.extend(lines)
            header_added = True
        else:
            # Skip leading comment/header lines for subsequent years
            i = 0
            while i < len(lines) and lines[i].lstrip().startswith("#"):
                i += 1
            combined_lines.extend(lines[i:])

        # Ensure there's a newline boundary between years
        if combined_lines and not combined_lines[-1].endswith("\n"):
            combined_lines.append("\n")

    out_path = os.path.join(OUT_DIR, f"{station}_stdmet_2021_2025.txt")
    if years_found > 0:
        with open(out_path, "w", encoding="utf-8") as f:
            f.writelines(combined_lines)

    return out_path, years_found


# main function
missing_all = []
for code in shefcodes:
    out_file, found = combine_station(code)
    if found == 0:
        missing_all.append(code)
        print(f"[MISS] {code}: no files found for 2021-2025")
    else:
        print(f"[OK]   {code}: combined {found} year(s) -> {out_file}")

print("\nDone.")
print("Stations with no data in 2021-2025:", len(missing_all))
if missing_all:
    print(missing_all[:50], "..." if len(missing_all) > 50 else "")

[OK]   SDBC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/SDBC1_stdmet_2021_2025.txt
[OK]   LJAC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/LJAC1_stdmet_2021_2025.txt
[OK]   AGXC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/AGXC1_stdmet_2021_2025.txt
[OK]   OHBC1: combined 3 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/OHBC1_stdmet_2021_2025.txt
[OK]   PRJC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/PRJC1_stdmet_2021_2025.txt
[OK]   PFDC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/PFDC1_stdmet_2021_2025.txt
[OK]   PFXC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/PFXC1_stdmet_2021_2025.txt
[OK]   PXAC1: combined 5 year(s) -> /content/drive/Shareddrives/CMPE_255/data/ndbc_wind_2021_2025/PXAC1_stdmet_2021_2025.txt


retreving the tides from co-ops

In [11]:
import os
import time
import math
import requests
import pandas as pd

MDAPI = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json"
DATA_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"

YEARS = range(2021, 2026)
DATUMS = ["MLLW", "MSL", "NAVD", "STND"]

TIMEOUT = 60
SLEEP_S = 0.2

PR_OUT_DIR = "/content/drive/Shareddrives/CMPE_255/data/coops_predictions_2021_2025"
os.makedirs(PR_OUT_DIR, exist_ok=True)

# ---------- helpers ----------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def month_ranges(years=YEARS):
    for y in years:
        for m in range(1, 13):
            begin = f"{y}{m:02d}01"
            end = f"{y+1}0101" if m == 12 else f"{y}{m+1:02d}01"
            yield begin, end

def fetch_predictions(station_id: str, begin_date: str, end_date: str,
                      datum: str | None, interval="h", session=None):
    sess = session or requests.Session()
    params = {
        "product": "predictions",
        "format": "json",
        "application": "predictions_only",
        "station": station_id,
        "begin_date": begin_date,
        "end_date": end_date,
        "time_zone": "gmt",
        "units": "metric",
        "interval": interval
    }
    if datum is not None:
        params["datum"] = datum

    r = sess.get(DATA_API, params=params, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    js = r.json()
    if "error" in js:
        return None
    rows = js.get("predictions")
    if not rows:
        return None
    return pd.DataFrame(rows)

def predictions_work_one_day(station_id: str, datum: str | None, interval="h", session=None):
    df = fetch_predictions(station_id, "20210101", "20210102", datum, interval=interval, session=session)
    return df is not None and not df.empty

def pick_predictions_datum(station_id: str, interval="h", session=None):
    # try standard datums then no-datum
    sess = session or requests.Session()
    for d in DATUMS:
        if predictions_work_one_day(station_id, d, interval=interval, session=sess):
            return d
        time.sleep(0.05)
    if predictions_work_one_day(station_id, None, interval=interval, session=sess):
        return None
    return "NO_DATUM_WORKED"

def build_candidates(session=None, restrict_state=None):
    sess = session or requests.Session()
    stations = sess.get(MDAPI, timeout=TIMEOUT).json().get("stations", [])
    candidates = []
    for s in stations:
        sid = s.get("id")
        if not sid:
            continue
        try:
            lat = float(s.get("lat"))
            lon = float(s.get("lng") or s.get("lon"))
        except (TypeError, ValueError):
            continue
        st = s.get("state")
        if restrict_state and st != restrict_state:
            continue
        candidates.append((sid, lat, lon, st))
    return candidates

def find_nearest_prediction_station(target_lat, target_lon, candidates, max_km=500,
                                   interval="h", session=None, pred_cache=None):
    sess = session or requests.Session()
    if pred_cache is None:
        pred_cache = {}

    ordered = sorted(candidates, key=lambda x: haversine_km(target_lat, target_lon, x[1], x[2]))

    for sid, lat, lon, st in ordered:
        dist = haversine_km(target_lat, target_lon, lat, lon)
        if dist > max_km:
            break

        if sid in pred_cache:
            ok, datum = pred_cache[sid]
        else:
            datum = pick_predictions_datum(sid, interval=interval, session=sess)
            ok = (datum != "NO_DATUM_WORKED")
            pred_cache[sid] = (ok, None if not ok else datum)

        ok, datum = pred_cache[sid]
        if ok:
            return sid, datum, dist

    return None, None, None

def map_targets_to_prediction_source(met_stations, max_km=500, interval="h", restrict_state=None):
    sess = requests.Session()
    candidates = build_candidates(session=sess, restrict_state=restrict_state)
    pred_cache = {}

    mapping = {}
    for s in met_stations:
        tid = str(s.get("id"))
        try:
            lat = float(s.get("lat"))
            lon = float(s.get("lng") or s.get("lon"))
        except (TypeError, ValueError):
            mapping[tid] = {"pred_station": None, "pred_datum": None, "distance_km": None}
            continue

        # First try predictions at the target itself (best)
        d = pick_predictions_datum(tid, interval=interval, session=sess)
        if d != "NO_DATUM_WORKED":
            mapping[tid] = {"pred_station": tid, "pred_datum": d, "distance_km": 0.0}
            continue

        # Otherwise, nearest station with predictions
        pid, pdatum, dist = find_nearest_prediction_station(
            lat, lon, candidates, max_km=max_km, interval=interval, session=sess, pred_cache=pred_cache
        )
        mapping[tid] = {"pred_station": pid, "pred_datum": pdatum, "distance_km": dist}

    return mapping

def download_predictions_2021_2025(pred_station: str, pred_datum: str | None, session=None):
    sess = session or requests.Session()
    frames = []
    for begin, end in month_ranges():
        df = fetch_predictions(pred_station, begin, end, pred_datum, interval="h", session=sess)
        if df is not None and not df.empty:
            frames.append(df)
        time.sleep(SLEEP_S)

    if not frames:
        return None

    out = pd.concat(frames, ignore_index=True)
    # ensure consistent columns: t, v
    return out

# # ---------- MAIN ----------
# def run_predictions_only(met_stations, max_km=500, restrict_state=None):
#     sess = requests.Session()

#     mapping = map_targets_to_prediction_source(
#         met_stations=met_stations,
#         max_km=max_km,
#         interval="h",
#         restrict_state=restrict_state
#     )

#     # cache by prediction station so we download each prediction series once
#     pred_series_cache = {}  # (pred_station, pred_datum) -> DataFrame

#     summary_rows = []

#     for target_id, m in mapping.items():
#         pred_station = m["pred_station"]
#         pred_datum = m["pred_datum"]
#         dist = m["distance_km"]

#         if not pred_station:
#             print(f"[MISS] {target_id} predictions: no station found within {max_km} km")
#             summary_rows.append({
#                 "target_station": target_id,
#                 "pred_status": "MISS",
#                 "pred_station": None,
#                 "pred_datum": None,
#                 "pred_distance_km": None,
#                 "file": None
#             })
#             continue

#         key = (pred_station, pred_datum)
#         if key not in pred_series_cache:
#             df = download_predictions_2021_2025(pred_station, pred_datum, session=sess)
#             pred_series_cache[key] = df

#         df = pred_series_cache[key]
#         if df is None or df.empty:
#             print(f"[MISS] {target_id} predictions via {pred_station} (datum={pred_datum})")
#             summary_rows.append({
#                 "target_station": target_id,
#                 "pred_status": "MISS",
#                 "pred_station": pred_station,
#                 "pred_datum": pred_datum,
#                 "pred_distance_km": dist,
#                 "file": None
#             })
#             continue

#         # Save one file per TARGET (even if shared source), so downstream is easy
#         suffix = f"src-{pred_station}_datum-{pred_datum if pred_datum is not None else 'none'}"
#         out_path = os.path.join(OUT_DIR, f"{target_id}_predictions_2021_2025_{suffix}.csv")
#         df.to_csv(out_path, index=False)

#         print(f"[OK]   {target_id} predictions via {pred_station} datum={pred_datum} dist_km={dist:.1f}")
#         summary_rows.append({
#             "target_station": target_id,
#             "pred_status": "OK",
#             "pred_station": pred_station,
#             "pred_datum": pred_datum,
#             "pred_distance_km": dist,
#             "file": out_path
#         })

#     summary_df = pd.DataFrame(summary_rows)
#     summary_path = os.path.join(OUT_DIR, "predictions_only_summary.csv")
#     summary_df.to_csv(summary_path, index=False)
#     print("\nWrote summary:", summary_path)
#     return mapping, summary_df

# # --------------------------
# # HOW TO CALL
# # --------------------------
# # met_ca is your list of station dicts (id, lat, lng/lon) from the met station query
# mapping, summary_df = run_predictions_only(met_ca, max_km=500, restrict_state=None)
# # (Optionally restrict candidates to CA: restrict_state="CA")

Download water level data

In [12]:
# =========================================================
# ADDITIONAL: WATER LEVEL (OBSERVED) DOWNLOAD (2021-2025)
# =========================================================

WL_OUT_DIR = "/content/drive/Shareddrives/CMPE_255/data/coops_water_level_2021_2025"
os.makedirs(WL_OUT_DIR, exist_ok=True)

def fetch_water_level(station_id: str, begin_date: str, end_date: str,
                      datum: str | None, interval="h", session=None):
    """
    Observed water level. interval="h" requests hourly (if supported).
    If interval is None, CO-OPS returns native (often 6-min).
    """
    sess = session or requests.Session()
    params = {
        "product": "water_level",
        "format": "json",
        "application": "water_level_pull",
        "station": station_id,
        "begin_date": begin_date,
        "end_date": end_date,
        "time_zone": "gmt",
        "units": "metric",
    }
    if datum is not None:
        params["datum"] = datum
    if interval is not None:
        params["interval"] = interval

    r = sess.get(DATA_API, params=params, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    js = r.json()
    if "error" in js:
        return None
    rows = js.get("data")
    if not rows:
        return None
    return pd.DataFrame(rows)

def water_level_work_one_day(station_id: str, datum: str | None, interval="h", session=None):
    df = fetch_water_level(station_id, "20210101", "20210102", datum, interval=interval, session=session)
    return df is not None and not df.empty

def pick_water_level_datum(station_id: str, interval="h", session=None):
    """
    Try datums. For water_level, datum is usually required; we’ll try fallbacks.
    If none work, return "NO_DATUM_WORKED".
    """
    sess = session or requests.Session()
    for d in DATUMS:
        if water_level_work_one_day(station_id, d, interval=interval, session=sess):
            return d
        time.sleep(0.05)

    # Some stations might not accept hourly interval; try native interval=None
    for d in DATUMS:
        if water_level_work_one_day(station_id, d, interval=None, session=sess):
            return d
        time.sleep(0.05)

    return "NO_DATUM_WORKED"

def water_level_station_ok(station_id: str, session=None):
    """Return (ok, datum, interval_used) for water_level."""
    sess = session or requests.Session()
    d = pick_water_level_datum(station_id, interval="h", session=sess)
    if d != "NO_DATUM_WORKED":
        # figure out whether hourly works
        if water_level_work_one_day(station_id, d, interval="h", session=sess):
            return True, d, "h"
        return True, d, None

    return False, None, None

def find_nearest_water_level_station(target_lat, target_lon, candidates,
                                     max_km=500, session=None, wl_cache=None):
    """
    Find nearest station that supports water_level. Returns (sid, datum, interval, dist_km).
    wl_cache: sid -> (ok, datum, interval)
    """
    sess = session or requests.Session()
    if wl_cache is None:
        wl_cache = {}

    ordered = sorted(candidates, key=lambda x: haversine_km(target_lat, target_lon, x[1], x[2]))

    for sid, lat, lon, st in ordered:
        dist = haversine_km(target_lat, target_lon, lat, lon)
        if dist > max_km:
            break

        if sid in wl_cache:
            ok, d, interval_used = wl_cache[sid]
        else:
            ok, d, interval_used = water_level_station_ok(sid, session=sess)
            wl_cache[sid] = (ok, d, interval_used)

        if ok:
            return sid, d, interval_used, dist

    return None, None, None, None

def map_targets_to_water_level_source(met_stations, max_km=500, restrict_state=None):
    """
    Similar to predictions mapping:
    - Try water_level at target
    - else nearest station with water_level
    """
    sess = requests.Session()
    candidates = build_candidates(session=sess, restrict_state=restrict_state)
    wl_cache = {}

    mapping = {}
    for s in met_stations:
        tid = str(s.get("id"))
        try:
            lat = float(s.get("lat"))
            lon = float(s.get("lng") or s.get("lon"))
        except (TypeError, ValueError):
            mapping[tid] = {"wl_station": None, "wl_datum": None, "wl_interval": None, "wl_distance_km": None}
            continue

        ok, d, interval_used = water_level_station_ok(tid, session=sess)
        if ok:
            mapping[tid] = {"wl_station": tid, "wl_datum": d, "wl_interval": interval_used, "wl_distance_km": 0.0}
            continue

        sid, d, interval_used, dist = find_nearest_water_level_station(
            lat, lon, candidates, max_km=max_km, session=sess, wl_cache=wl_cache
        )
        mapping[tid] = {"wl_station": sid, "wl_datum": d, "wl_interval": interval_used, "wl_distance_km": dist}

    return mapping

def download_water_level_2021_2025(wl_station: str, wl_datum: str, wl_interval: str | None, session=None):
    sess = session or requests.Session()
    frames = []
    for begin, end in month_ranges():
        df = fetch_water_level(wl_station, begin, end, wl_datum, interval=wl_interval, session=sess)
        if df is not None and not df.empty:
            frames.append(df)
        time.sleep(SLEEP_S)

    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)

def run_predictions_and_water_level(met_stations, max_km=500, restrict_state=None):
    """
    Runs BOTH:
      - predictions mapping+download (your existing behavior)
      - water_level mapping+download (new)
    Saves per-target files for both products, and combined summaries.
    """
    sess = requests.Session()

    # 1) Predictions (your existing mapping)
    pred_mapping = map_targets_to_prediction_source(
        met_stations=met_stations,
        max_km=max_km,
        interval="h",
        restrict_state=restrict_state
    )

    # 2) Water level mapping (new)
    wl_mapping = map_targets_to_water_level_source(
        met_stations=met_stations,
        max_km=max_km,
        restrict_state=restrict_state
    )

    # Caches so we download each source station once
    pred_cache = {}  # (pred_station, pred_datum) -> DataFrame
    wl_cache = {}    # (wl_station, wl_datum, wl_interval) -> DataFrame

    rows = []

    for s in met_stations:
        target_id = str(s.get("id"))

        # ---- Predictions ----
        pm = pred_mapping.get(target_id, {})
        pred_station = pm.get("pred_station")
        pred_datum = pm.get("pred_datum")
        pred_dist = pm.get("distance_km")

        pred_file = None
        pred_status = "MISS"
        if pred_station:
            pkey = (pred_station, pred_datum)
            if pkey not in pred_cache:
                pred_cache[pkey] = download_predictions_2021_2025(pred_station, pred_datum, session=sess)

            pdf = pred_cache[pkey]
            if pdf is not None and not pdf.empty:
                suffix = f"src-{pred_station}_datum-{pred_datum if pred_datum is not None else 'none'}"
                pred_file = os.path.join(PR_OUT_DIR, f"{target_id}_predictions_2021_2025_{suffix}.csv")
                pdf.to_csv(pred_file, index=False)
                pred_status = "OK"
                print(f"[OK]   {target_id} predictions via {pred_station} datum={pred_datum} dist_km={pred_dist:.1f}")
            else:
                print(f"[MISS] {target_id} predictions via {pred_station} (datum={pred_datum})")
        else:
            print(f"[MISS] {target_id} predictions: no station found within {max_km} km")

        # ---- Water Level ----
        wm = wl_mapping.get(target_id, {})
        wl_station = wm.get("wl_station")
        wl_datum = wm.get("wl_datum")
        wl_interval = wm.get("wl_interval")
        wl_dist = wm.get("wl_distance_km")

        wl_file = None
        wl_status = "MISS"
        if wl_station and wl_datum:
            wkey = (wl_station, wl_datum, wl_interval)
            if wkey not in wl_cache:
                wl_cache[wkey] = download_water_level_2021_2025(wl_station, wl_datum, wl_interval, session=sess)

            wdf = wl_cache[wkey]
            if wdf is not None and not wdf.empty:
                suffix = f"src-{wl_station}_datum-{wl_datum}_interval-{wl_interval if wl_interval else 'native'}"
                wl_file = os.path.join(WL_OUT_DIR, f"{target_id}_water_level_2021_2025_{suffix}.csv")
                wdf.to_csv(wl_file, index=False)
                wl_status = "OK"
                print(f"[OK]   {target_id} water_level via {wl_station} datum={wl_datum} interval={wl_interval} dist_km={wl_dist:.1f}")
            else:
                print(f"[MISS] {target_id} water_level via {wl_station} (datum={wl_datum})")
        else:
            print(f"[MISS] {target_id} water_level: no station found within {max_km} km")

        rows.append({
            "target_station": target_id,

            "pred_status": pred_status,
            "pred_station": pred_station,
            "pred_datum": pred_datum,
            "pred_distance_km": pred_dist,
            "pred_file": pred_file,

            "wl_status": wl_status,
            "wl_station": wl_station,
            "wl_datum": wl_datum,
            "wl_interval": wl_interval,
            "wl_distance_km": wl_dist,
            "wl_file": wl_file,
        })

    summary_df = pd.DataFrame(rows)
    summary_path = os.path.join("coops_pred_wl_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print("\nWrote combined summary:", summary_path)

    return pred_mapping, wl_mapping, summary_df


# --------------------------
# HOW TO CALL (replaces your last line)
# --------------------------
pred_mapping, wl_mapping, summary_df = run_predictions_and_water_level(met_ca, max_km=500, restrict_state=None)

[OK]   9410170 predictions via 9410170 datum=MLLW dist_km=0.0
[OK]   9410170 water_level via 9410170 datum=MLLW interval=h dist_km=0.0
[OK]   9410230 predictions via 9410230 datum=MLLW dist_km=0.0
[OK]   9410230 water_level via 9410230 datum=MLLW interval=h dist_km=0.0
[OK]   9410647 predictions via 9410660 datum=MLLW dist_km=2.4
[OK]   9410647 water_level via 9410660 datum=MLLW interval=h dist_km=2.4
[OK]   9410660 predictions via 9410660 datum=MLLW dist_km=0.0
[OK]   9410660 water_level via 9410660 datum=MLLW interval=h dist_km=0.0
[OK]   9410665 predictions via 9410660 datum=MLLW dist_km=8.1
[OK]   9410665 water_level via 9410660 datum=MLLW interval=h dist_km=8.1
[OK]   9410666 predictions via 9410660 datum=MLLW dist_km=3.3
[OK]   9410666 water_level via 9410660 datum=MLLW interval=h dist_km=3.3
[OK]   9410670 predictions via 9410660 datum=MLLW dist_km=6.0
[OK]   9410670 water_level via 9410660 datum=MLLW interval=h dist_km=6.0
[OK]   9410690 predictions via 9410660 datum=MLLW dist_